# AI-Powered Text Adventure Game Demo

这是一个交互式的文字冒险游戏演示。
- 玩家可以在任何时刻执行任意操作
- 或者选择让AI自动继续故事
- 游戏状态会被自动保存和管理

## 1. 安装和导入

In [ ]:
# 安装依赖（首次运行）
import subprocess
import sys

def install_requirements():
    try:
        import openai
        print("✓ openai 已安装")
    except:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "openai"])
    
    try:
        import ipywidgets
        print("✓ ipywidgets 已安装")
    except:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "ipywidgets"])

install_requirements()
print("\n依赖检查完成！")

In [ ]:
import sys
import os
from pathlib import Path
from dotenv import load_dotenv

# 添加game模块到路径
sys.path.insert(0, str(Path.cwd()))

# 加载.env文件中的API密钥
load_dotenv()

# 导入游戏模块
from game import create_generator, TextAdventureGame

print("✓ 导入成功！")
print(f"当前工作目录：{Path.cwd()}")

## 2. 配置AI后端

选择你想使用的AI模型

In [ ]:
import ipywidgets as widgets
from IPython.display import display, HTML

# 创建后端选择器
backend_dropdown = widgets.Dropdown(
    options=[
        ('OpenAI (GPT-4o-mini)', 'openai_mini'),
        ('OpenAI (GPT-3.5-turbo)', 'openai_35'),
        ('Claude', 'claude'),
        ('本地模型 (Ollama)', 'local')
    ],
    value='openai_mini',
    description='选择AI背景:',
    style={'description_width': '100px'}
)

display(backend_dropdown)
print("\n提示：确保你的API密钥已设置在.env文件中")

In [ ]:
# 初始化游戏生成器
backend_choice = backend_dropdown.value

try:
    if backend_choice == 'openai_mini':
        generator = create_generator(backend="openai", model_name="gpt-4o-mini")
        print("✓ 使用 OpenAI GPT-4o-mini")
    elif backend_choice == 'openai_35':
        generator = create_generator(backend="openai", model_name="gpt-3.5-turbo")
        print("✓ 使用 OpenAI GPT-3.5-turbo")
    elif backend_choice == 'claude':
        generator = create_generator(backend="claude", model_name="claude-3-5-sonnet-20241022")
        print("✓ 使用 Claude 3.5 Sonnet")
    elif backend_choice == 'local':
        generator = create_generator(backend="local", model_name="llama3")
        print("✓ 使用本地Llama3模型")
    
    # 初始化游戏
    game = TextAdventureGame(generator, save_dir="./output")
    print("✓ 游戏引擎初始化成功！")
except Exception as e:
    print(f"❌ 初始化失败: {e}")
    print("\n提示：")
    print("- 检查API密钥是否正确设置在.env文件中")
    print("- 如果使用本地模型，确保Ollama已启动")

## 3. 开始游戏

In [ ]:
# 生成开场故事
print("🎮 正在生成开场故事...\n")

try:
    opening_story = game.initialize_story()
    print(opening_story)
    print("\n" + "="*60 + "\n")
except Exception as e:
    print(f"❌ 生成失败: {e}")

## 4. 玩家交互循环

在下方输入你的操作，或选择'自动继续'让AI继续生成故事

In [ ]:
# 创建交互界面
action_input = widgets.Text(
    value='',
    placeholder='输入你的操作（例如：向前走，查看四周）',
    description='你的操作:',
    style={'description_width': '100px'}
)

auto_continue_button = widgets.Button(
    description='🔄 自动继续',
    button_style='info',
    tooltip='让AI自动继续故事'
)

execute_button = widgets.Button(
    description='▶️ 执行操作',
    button_style='success',
    tooltip='执行你的操作'
)

save_button = widgets.Button(
    description='💾 保存游戏',
    button_style='warning',
    tooltip='保存游戏进度'
)

output_area = widgets.Output()

def on_execute_click(b):
    with output_area:
        action = action_input.value.strip()
        if not action:
            print("⚠️ 请输入你的操作")
            return
        
        print(f"\n▶️ 你的操作: {action}\n")
        print("🤖 AI 生成中...\n")
        
        try:
            story = game.process_action(action)
            print(story)
            print("\n" + "="*60)
            action_input.value = ''
        except Exception as e:
            print(f"❌ 错误: {e}")

def on_auto_continue(b):
    with output_area:
        print("\n🔄 自动继续...\n")
        print("🤖 AI 生成中...\n")
        
        try:
            story = game.continue_story()
            print(story)
            print("\n" + "="*60)
        except Exception as e:
            print(f"❌ 错误: {e}")

def on_save_click(b):
    with output_area:
        filepath = game.save_game()
        print(f"✓ 游戏已保存到: {filepath}")

execute_button.on_click(on_execute_click)
auto_continue_button.on_click(on_auto_continue)
save_button.on_click(on_save_click)

# 显示控件
display(HTML("<h3>游戏操作</h3>"))
display(action_input)
display(widgets.HBox([execute_button, auto_continue_button, save_button]))
display(output_area)

print("✓ 交互界面已加载！开始输入你的操作吧！")

## 5. 游戏状态查看

In [ ]:
# 查看当前游戏状态
import json

status = game.get_status()
print("📊 游戏状态：")
print(json.dumps(status, indent=2, ensure_ascii=False))

print(f"\n📝 故事事件数: {len(game.game_log)}")
print(f"📍 当前位置: {game.state.location}")
print(f"🎒 物品数: {len(game.state.inventory)}")
print(f"🧑 已知NPC: {len(game.state.npcs)}")

## 6. 查看已保存的游戏

In [ ]:
# 列出所有保存的游戏
saves = game.list_saves()

if saves:
    print(f"已保存的游戏 ({len(saves)} 个):")
    for i, save_file in enumerate(saves):
        print(f"  {i+1}. {save_file}")
else:
    print("还没有保存任何游戏")

## 提示

- 🎮 **玩法**: 输入任何操作描述，游戏会根据你的操作生成故事
- 🔄 **自动继续**: 如果不输入操作，点"自动继续"让AI继续生成
- 💾 **保存游戏**: 随时点"保存游戏"存档当前进度
- 📊 **查看状态**: 执行第5格的代码查看当前游戏状态
- ⚠️ **成本提示**: 每次AI调用都会消耗tokens，注意API使用成本